# Raw/Bronze → Silver | State of Data Brasil

Notebook simplificado para execução no **AWS Glue Notebook**, utilizando **PySpark**.

## Objetivo

Ler as três pesquisas já disponíveis na camada Raw/Bronze, manter somente as variáveis necessárias
para responder às perguntas de negócio, harmonizar os três anos e gravar uma camada Silver única.

### Entrada

- `workspace.tb_survey_2023`
- `workspace.tb_survey_2024`
- `workspace.tb_survey_2025`

### Saída no S3

```text
silver/
└── state_of_data/
    ├── ano=2023/
    ├── ano=2024/
    └── ano=2025/
```

### Perguntas que a Silver deve permitir responder

- Como está estruturado o mercado brasileiro de Dados?
- Quais perfis profissionais são mais valorizados?
- Qual é o cenário de diversidade de gênero?
- Quais tecnologias apresentam maior adoção?
- Qual é o índice de adoção de IA e seu impacto?
- Existem diferenças entre regiões, senioridades ou modelos de trabalho?
- Quais oportunidades e desafios existem para empresas que desejam investir em Dados e IA?

> A Silver permanece no nível de respondente. Rankings, percentuais, médias e indicadores serão construídos na Gold.

## Estrutura do notebook

1. Inicialização do Glue  
2. Parâmetros  
3. Leitura das tabelas Bronze  
4. Configuração dos campos por ano  
5. Funções simples de apoio  
6. Seleção e harmonização dos três anos  
7. Limpeza básica  
8. Variáveis derivadas necessárias  
9. União e deduplicação simples  
10. Controle de qualidade  
11. Escrita no S3  
12. Catalogação da Silver no Glue Data Catalog  
13. Validação final

## 1. Inicialização do AWS Glue

In [1]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

import boto3
import re
import unicodedata

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Spark:", spark.version)
print("Sessão pronta.")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 5a92c394-9dcb-4413-ad8c-814f7ff61618
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 5a92c394-9dcb-4413-ad8c-814f7ff61618 to get into ready status...
Session 5a92c394-9dcb-4413-ad8c-814f7ff61618 has been created.
Spark: 3.3.0-amzn-1
Sessão pronta.


## 2. Parâmetros do projeto

Não é necessário configurar credenciais no notebook.
A execução utiliza a role associada à sessão do AWS Glue.

In [ ]:
DATABASE = "workspace"

TABELAS_BRONZE = {
    2023: "tb_survey_2023",
    2024: "tb_survey_2024",
    2025: "tb_survey_2025",
}

BUCKET = "tech-challenge-fase3-015006598133"
SILVER_PATH = f"s3://{BUCKET}/silver/state_of_data/"
SILVER_TABLE = "tb_state_of_data_silver"

print("Silver:", SILVER_PATH)

Silver: s3://tech-challenge-fase3-015006598133/silver/state_of_data/


## 3. Leitura das tabelas Bronze pelo Glue Data Catalog

In [ ]:
def ler_tabela(table_name):
    return glueContext.create_data_frame_from_catalog(
        database=DATABASE,
        table_name=table_name
    )


bronze = {
    ano: ler_tabela(tabela)
    for ano, tabela in TABELAS_BRONZE.items()
}


for ano, df in bronze.items():
    print(
        f"{ano}: "
        f"{df.count()} linhas | "
        f"{len(df.columns)} colunas"
    )

2023: 5293 linhas | 399 colunas
2024: 5217 linhas | 403 colunas
2025: 3495 linhas | 388 colunas
/opt/amazon/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.


## 4. Configuração dos campos por ano

As pesquisas mudaram os nomes das perguntas ao longo dos anos.

Para evitar um mapeamento longo com os textos completos das perguntas, o notebook utiliza apenas
o **código da pergunta**, como `P2_f`, `2.f`, `4.d`, etc.

Isso também reduz problemas causados pelo Glue ao simplificar nomes de colunas durante a catalogação.

### Campos multisseleção

Linguagens, bancos, cloud, BI e uso de IA podem aparecer como:

- uma coluna consolidada; ou
- várias colunas binárias, uma para cada opção.

A função do próximo bloco trata os dois formatos.

In [ ]:
CONFIG = {
    2023: {
        "respondent_id": "p0",
        "faixa_idade": "p1a1",
        "genero": "p1b",
        "uf": "p1i1",
        "regiao": "p1i2",
        "nivel_ensino": "p1l",
        "area_formacao": "p1m",

        "situacao_trabalho": "p2a",
        "setor": "p2b",
        "cargo_atual": "p2f",
        "senioridade": "p2g",
        "faixa_salarial": "p2h",
        "tempo_experiencia_dados": "p2i",
        "modelo_trabalho": "p2r",

        "desafios_gestor": "p3d",
        "ia_prioridade_empresa": "p3e",
        "ia_resultados_empresa": None,
        "ia_motivos_nao_uso_empresa": "p3g",

        "linguagens_uso": "p4d",
        "bancos_dados_uso": "p4g",
        "cloud_uso": "p4h",
        "ferramentas_bi_uso": "p4j",
        "usa_ia_llm": "p4m",
    },

    2024: {
        "respondent_id": "0a",
        "faixa_idade": "1a1",
        "genero": "1b",
        "uf": "1i1",
        "regiao": "1i2",
        "nivel_ensino": "1l",
        "area_formacao": "1m",

        "situacao_trabalho": "2a",
        "setor": "2b",
        "cargo_atual": "2f",
        "senioridade": "2g",
        "faixa_salarial": "2h",
        "tempo_experiencia_dados": "2i",
        "modelo_trabalho": "2r",

        "desafios_gestor": "3d",
        "ia_prioridade_empresa": "3e",
        "ia_resultados_empresa": None,
        "ia_motivos_nao_uso_empresa": "3g",

        "linguagens_uso": "4d",
        "bancos_dados_uso": "4g",
        "cloud_uso": "4h",
        "ferramentas_bi_uso": "4j",
        "usa_ia_llm": "4m",
    },

    2025: {
        "respondent_id": "0a",
        "faixa_idade": "1a1",
        "genero": "1b",
        "uf": "1i1",
        "regiao": "1i2",
        "nivel_ensino": "1l",
        "area_formacao": "1m",

        "situacao_trabalho": "2a",
        "setor": "2b",
        "cargo_atual": "2f",
        "senioridade": "2g",
        "faixa_salarial": "2h",
        "tempo_experiencia_dados": "2i",
        "modelo_trabalho": "2q",

        "desafios_gestor": "3d",
        "ia_prioridade_empresa": "3e",
        "ia_resultados_empresa": "3g",
        "ia_motivos_nao_uso_empresa": "3h",

        "linguagens_uso": "4c",
        "bancos_dados_uso": "4d",
        "cloud_uso": "4e",
        "ferramentas_bi_uso": "4g",
        "usa_ia_llm": "4j",
    },
}


CAMPOS_MULTI = {
    "linguagens_uso",
    "bancos_dados_uso",
    "cloud_uso",
    "ferramentas_bi_uso",
    "usa_ia_llm",
    "desafios_gestor",
    "ia_motivos_nao_uso_empresa",
}

## 5. Funções simples de apoio

Este é o único bloco de compatibilidade de schema.

Ele:

1. normaliza o nome das colunas;
2. procura a pergunta pelo código;
3. para campos multisseleção, combina as opções marcadas.

Não há `raise`, hash, auditoria complexa ou dezenas de validações.

In [ ]:
def normalizar_nome(nome):
    nome = unicodedata.normalize(
        "NFKD",
        str(nome)
    )

    nome = "".join(
        c for c in nome
        if not unicodedata.combining(c)
    )

    return re.sub(
        r"[^a-z0-9]+",
        "",
        nome.lower()
    )


def candidatos_por_codigo(df, codigo):
    if codigo is None:
        return []

    return [
        coluna
        for coluna in df.columns
        if normalizar_nome(coluna).startswith(codigo)
    ]


def coluna_simples(df, codigo):
    candidatos = candidatos_por_codigo(
        df,
        codigo
    )

    if not candidatos:
        return None

    return sorted(
        candidatos,
        key=lambda c: len(
            normalizar_nome(c)
        )
    )[0]


def resposta_multiselect(df, codigo):
    candidatos = candidatos_por_codigo(
        df,
        codigo
    )

    if not candidatos:
        return F.lit(None).cast("string")

    opcoes = [
        coluna
        for coluna in candidatos
        if re.match(
            rf"^{codigo}\d+",
            normalizar_nome(coluna)
        )
    ]

    if not opcoes:
        consolidada = sorted(
            candidatos,
            key=lambda c: len(
                normalizar_nome(c)
            )
        )[0]

        return F.col(
            f"`{consolidada}`"
        ).cast("string")

    respostas = []

    for coluna in opcoes:
        nome_norm = normalizar_nome(
            coluna
        )

        rotulo_norm = re.sub(
            rf"^{codigo}\d+",
            "",
            nome_norm
        )

        rotulo = re.sub(
            r"^[^a-zA-ZÀ-ÿ]*",
            "",
            re.sub(
                rf"(?i)^.*?{codigo[-1]}[._-]*\d+[._-]*",
                "",
                str(coluna)
            )
        ).strip()

        if not rotulo:
            rotulo = rotulo_norm

        valor = F.lower(
            F.trim(
                F.col(
                    f"`{coluna}`"
                ).cast("string")
            )
        )

        selecionado = valor.isin(
            "1",
            "1.0",
            "true",
            "sim",
            "yes"
        )

        respostas.append(
            F.when(
                selecionado,
                F.lit(rotulo)
            )
        )

    combinado = F.concat_ws(
        "|||",
        *respostas
    )

    return F.when(
        combinado != "",
        combinado
    ).otherwise(
        F.lit(None).cast("string")
    )


def selecionar_ano(df, ano):
    expressoes = []

    for campo, codigo in CONFIG[ano].items():

        if codigo is None:
            expressoes.append(
                F.lit(None)
                .cast("string")
                .alias(campo)
            )

        elif campo in CAMPOS_MULTI:
            expressoes.append(
                resposta_multiselect(
                    df,
                    codigo
                ).alias(campo)
            )

        else:
            origem = coluna_simples(
                df,
                codigo
            )

            if origem is None:
                expressoes.append(
                    F.lit(None)
                    .cast("string")
                    .alias(campo)
                )
            else:
                expressoes.append(
                    F.col(
                        f"`{origem}`"
                    )
                    .cast("string")
                    .alias(campo)
                )

    return (
        df
        .select(*expressoes)
        .withColumn(
            "ano",
            F.lit(ano).cast("int")
        )
    )

## 6. Seleção e harmonização dos três anos

Neste ponto cada pesquisa passa a ter o mesmo conjunto de colunas.

Campos inexistentes em determinado ano recebem `NULL`.

In [6]:
silver_anos = {
    ano: selecionar_ano(
        bronze[ano],
        ano
    )
    for ano in [2023, 2024, 2025]
}


for ano, df in silver_anos.items():
    print(
        f"{ano}: "
        f"{df.count()} linhas | "
        f"{len(df.columns)} colunas"
    )

2023: 5293 linhas | 24 colunas
2024: 5217 linhas | 24 colunas
2025: 3495 linhas | 24 colunas


## 7. Limpeza básica

In [7]:
VALORES_NULOS = [
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "-",
]


def limpar_textos(df):
    for coluna, tipo in df.dtypes:

        if tipo == "string":

            valor = F.regexp_replace(
                F.trim(
                    F.col(coluna)
                ),
                r"\s+",
                " "
            )

            df = df.withColumn(
                coluna,
                F.when(
                    valor.isNull()
                    | F.lower(valor).isin(
                        VALORES_NULOS
                    ),
                    F.lit(None)
                ).otherwise(
                    valor
                )
            )

    return df


silver_anos = {
    ano: limpar_textos(df)
    for ano, df in silver_anos.items()
}

## 8. Variáveis derivadas necessárias para análise

Serão criadas somente as derivações úteis para as perguntas de negócio:

- `grupo_cargo`
- `senioridade_grupo`
- `modelo_trabalho_grupo`
- `salario_estimado`
- `ia_adocao`

As respostas originais continuam disponíveis.

In [ ]:
def adicionar_variaveis_derivadas(df):

    # 1. Grupo de cargo
    cargo = F.lower(
        F.coalesce(
            F.col("cargo_atual"),
            F.lit("")
        )
    )

    df = df.withColumn(
        "grupo_cargo",
        F.when(
            cargo.contains("cientista"),
            "Ciência de Dados"
        )
        .when(
            cargo.contains("engenheiro de dados")
            | cargo.contains("arquiteto de dados"),
            "Engenharia de Dados"
        )
        .when(
            cargo.contains("analytics engineer"),
            "Analytics Engineering"
        )
        .when(
            cargo.contains("analista de dados"),
            "Análise de Dados"
        )
        .when(
            cargo.contains("business intelligence")
            | cargo.contains("analista de bi"),
            "Business Intelligence"
        )
        .when(
            cargo.contains("machine learning")
            | cargo.contains("ml engineer")
            | cargo.contains("ai engineer"),
            "Machine Learning / IA"
        )
        .when(
            cargo.contains("gerente")
            | cargo.contains("coordenador")
            | cargo.contains("diretor")
            | cargo.contains("head"),
            "Gestão / Liderança"
        )
        .when(
            F.col("cargo_atual").isNull(),
            F.lit(None).cast("string")
        )
        .otherwise(
            "Outros"
        )
    )


    # 2. Senioridade
    senioridade = F.lower(
        F.coalesce(
            F.col("senioridade"),
            F.lit("")
        )
    )

    df = df.withColumn(
        "senioridade_grupo",
        F.when(
            senioridade.contains("jun"),
            "Júnior"
        )
        .when(
            senioridade.contains("pleno"),
            "Pleno"
        )
        .when(
            senioridade.contains("senior")
            | senioridade.contains("sênior"),
            "Sênior"
        )
        .when(
            senioridade.contains("especialista")
            | senioridade.contains("staff"),
            "Especialista / Staff+"
        )
        .when(
            F.col("senioridade").isNull(),
            F.lit(None).cast("string")
        )
        .otherwise(
            F.initcap(
                F.col("senioridade")
            )
        )
    )


    # 3. Modelo de trabalho
    modelo = F.lower(
        F.coalesce(
            F.col("modelo_trabalho"),
            F.lit("")
        )
    )

    df = df.withColumn(
        "modelo_trabalho_grupo",
        F.when(
            modelo.contains("remot"),
            "Remoto"
        )
        .when(
            modelo.contains("híbr")
            | modelo.contains("hibr"),
            "Híbrido"
        )
        .when(
            modelo.contains("presencial"),
            "Presencial"
        )
        .when(
            F.col("modelo_trabalho").isNull(),
            F.lit(None).cast("string")
        )
        .otherwise(
            "Outro"
        )
    )


    # 4. Salário estimado
    salario = F.col(
        "faixa_salarial"
    )

    limite_inferior_txt = F.regexp_extract(
        salario,
        r"(?i)de\s+R\$\s*([\d\.]+)",
        1
    )

    limite_superior_txt = F.regexp_extract(
        salario,
        r"(?i)a\s+R\$\s*([\d\.]+)",
        1
    )

    acima_txt = F.regexp_extract(
        salario,
        r"(?i)acima\s+de\s+R\$\s*([\d\.]+)",
        1
    )

    menos_txt = F.regexp_extract(
        salario,
        r"(?i)menos\s+de\s+R\$\s*([\d\.]+)",
        1
    )

    inferior = F.regexp_replace(
        limite_inferior_txt,
        r"\.",
        ""
    ).cast("double")

    superior = F.regexp_replace(
        limite_superior_txt,
        r"\.",
        ""
    ).cast("double")

    acima = F.regexp_replace(
        acima_txt,
        r"\.",
        ""
    ).cast("double")

    menos = F.regexp_replace(
        menos_txt,
        r"\.",
        ""
    ).cast("double")

    df = df.withColumn(
        "salario_estimado",
        F.when(
            (inferior > 0)
            & (superior >= inferior),
            (inferior + superior) / 2
        )
        .when(
            acima > 0,
            acima
        )
        .when(
            menos > 0,
            menos / 2
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )


    # 5. Adoção de IA
    ia = F.lower(
        F.coalesce(
            F.col("usa_ia_llm"),
            F.lit("")
        )
    )

    df = df.withColumn( 
        "ia_adocao",

        F.when(
            F.col("usa_ia_llm").isNull(),
            "Não utiliza"
        )

        .when(
            ia.contains("não utiliz")
            | ia.contains("nao utiliz")
            | ia.contains("não uso")
            | ia.contains("nao uso"),
            "Não utiliza"
        )

        .otherwise(
            "Utiliza"
        )
    )

    return df


silver_anos = {
    ano: adicionar_variaveis_derivadas(df)
    for ano, df in silver_anos.items()
}

In [ ]:
# VALIDACAO IA
ia = F.lower(
    F.coalesce(
        F.col("usa_ia_llm"),
        F.lit("")
    )
)

df = df.withColumn(
    "ia_adocao",

    F.when(
        F.col("usa_ia_llm").isNull(),
        "Não utiliza"
    )

    .when(
        ia.contains("não utiliz")
        | ia.contains("nao utiliz")
        | ia.contains("não uso")
        | ia.contains("nao uso"),
        "Não utiliza"
    )

    .otherwise(
        "Utiliza"
    )
)

## 9. União e deduplicação simples

A deduplicação será feita apenas com `dropDuplicates()`.

Isso remove linhas totalmente iguais sem utilizar hash ou regras adicionais.

In [ ]:
df_silver = (
    silver_anos[2023]
    .unionByName(
        silver_anos[2024],
        allowMissingColumns=True
    )
    .unionByName(
        silver_anos[2025],
        allowMissingColumns=True
    )
    .dropDuplicates()
    .cache()
)

print("Total Silver:", df_silver.count())

df_silver.groupBy("ano").count().orderBy("ano").show()

Total Silver: 14002
+----+-----+
| ano|count|
+----+-----+
|2023| 5293|
|2024| 5215|
|2025| 3494|
+----+-----+


## 10. Controle de qualidade

O controle é propositalmente simples:

- quantidade de registros por ano;
- quantidade de IDs distintos;
- quantidade de valores nulos nos principais campos;
- amostra dos dados tratados.

In [ ]:
CAMPOS_QUALIDADE = [
    "respondent_id",
    "genero",
    "regiao",
    "cargo_atual",
    "senioridade",
    "faixa_salarial",
    "modelo_trabalho",
    "linguagens_uso",
    "usa_ia_llm",
]


for ano in [2023, 2024, 2025]:

    df_ano = df_silver.filter(
        F.col("ano") == ano
    )

    total = df_ano.count()

    ids = df_ano.select(
        "respondent_id"
    ).distinct().count()

    print("=" * 70)
    print(f"ANO {ano}")
    print("Registros:", total)
    print("IDs distintos:", ids)

    nulos = df_ano.select(
        [
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in CAMPOS_QUALIDADE
        ]
    )

    nulos.show(truncate=False)

ANO 2023
Registros: 5293
IDs distintos: 5293
+-------------+------+------+-----------+-----------+--------------+---------------+--------------+----------+
|respondent_id|genero|regiao|cargo_atual|senioridade|faixa_salarial|modelo_trabalho|linguagens_uso|usa_ia_llm|
+-------------+------+------+-----------+-----------+--------------+---------------+--------------+----------+
|0            |0     |124   |1436       |1436       |540           |540            |1521          |1521      |
+-------------+------+------+-----------+-----------+--------------+---------------+--------------+----------+

ANO 2024
Registros: 5215
IDs distintos: 5215
+-------------+------+------+-----------+-----------+--------------+---------------+--------------+----------+
|respondent_id|genero|regiao|cargo_atual|senioridade|faixa_salarial|modelo_trabalho|linguagens_uso|usa_ia_llm|
+-------------+------+------+-----------+-----------+--------------+---------------+--------------+----------+
|0            |0     

In [ ]:
df_silver.select(
    "ano",
    "respondent_id",
    "genero",
    "regiao",
    "cargo_atual",
    "grupo_cargo",
    "senioridade_grupo",
    "faixa_salarial",
    "salario_estimado",
    "modelo_trabalho_grupo",
    "ia_adocao",
).show(20, truncate=False)

+----+--------------------------------+---------+------------+------------------------------------------------+---------------------+---------------------+--------------------------------+----------------+---------------------+---------+
|ano |respondent_id                   |genero   |regiao      |cargo_atual                                     |grupo_cargo          |senioridade_grupo    |faixa_salarial                  |salario_estimado|modelo_trabalho_grupo|ia_adocao|
+----+--------------------------------+---------+------------+------------------------------------------------+---------------------+---------------------+--------------------------------+----------------+---------------------+---------+
|2025|nqxsamjcc5h9sosvntfnqxslws38ddpw|Masculino|Sudeste     |null                                            |null                 |null                 |de R$ 30.001/mês a R$ 40.000/mês|35000.5         |Híbrido              |null     |
|2025|rfb2kr8qkoc0kjoyte1sirfb2kr80ujh|Masculino

## 11. Escrita da camada Silver no S3

O `overwrite` permite reexecutar o notebook sem acumular versões anteriores.

A coluna `ano` será utilizada como partição.

In [ ]:
(
    df_silver
    .write
    .mode("overwrite")
    .partitionBy("ano")
    .parquet(SILVER_PATH)
)

print("Silver gravada em:", SILVER_PATH)

Silver gravada em: s3://tech-challenge-fase3-015006598133/silver/state_of_data/


## 12. Catalogação da Silver no Glue Data Catalog

Este bloco registra a saída Parquet como:

```text
workspace.tb_state_of_data_silver
```

e registra as três partições:

```text
ano=2023
ano=2024
ano=2025
```

O bloco é separado do ETL para manter o tratamento simples e legível.

In [13]:
glue = boto3.client(
    "glue",
    region_name="us-east-1"
)


def tipo_glue(data_type):
    if isinstance(data_type, T.StringType):
        return "string"
    if isinstance(data_type, T.IntegerType):
        return "int"
    if isinstance(data_type, T.LongType):
        return "bigint"
    if isinstance(data_type, T.DoubleType):
        return "double"
    if isinstance(data_type, T.FloatType):
        return "float"
    if isinstance(data_type, T.BooleanType):
        return "boolean"
    if isinstance(data_type, T.TimestampType):
        return "timestamp"

    return "string"


colunas_catalogo = [
    {
        "Name": campo.name,
        "Type": tipo_glue(
            campo.dataType
        )
    }
    for campo in df_silver.schema.fields
    if campo.name != "ano"
]


storage_descriptor = {
    "Columns": colunas_catalogo,
    "Location": SILVER_PATH,
    "InputFormat": (
        "org.apache.hadoop.hive.ql.io.parquet."
        "MapredParquetInputFormat"
    ),
    "OutputFormat": (
        "org.apache.hadoop.hive.ql.io.parquet."
        "MapredParquetOutputFormat"
    ),
    "SerdeInfo": {
        "SerializationLibrary": (
            "org.apache.hadoop.hive.ql.io.parquet.serde."
            "ParquetHiveSerDe"
        )
    }
}


table_input = {
    "Name": SILVER_TABLE,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "parquet",
        "EXTERNAL": "TRUE"
    },
    "PartitionKeys": [
        {
            "Name": "ano",
            "Type": "int"
        }
    ],
    "StorageDescriptor": storage_descriptor
}


try:
    glue.get_table(
        DatabaseName=DATABASE,
        Name=SILVER_TABLE
    )

    glue.update_table(
        DatabaseName=DATABASE,
        TableInput=table_input
    )

    print(
        "Tabela atualizada no catálogo."
    )

except glue.exceptions.EntityNotFoundException:

    glue.create_table(
        DatabaseName=DATABASE,
        TableInput=table_input
    )

    print(
        "Tabela criada no catálogo."
    )

{'ResponseMetadata': {'RequestId': '4ad0e136-ae80-4718-b566-2ed39a2433e7', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 11 Aug 2026 01:33:12 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': '4ad0e136-ae80-4718-b566-2ed39a2433e7', 'cache-control': 'no-cache'}, 'RetryAttempts': 0}}
Tabela criada no catálogo.


In [14]:
for ano in [2023, 2024, 2025]:

    partition_input = {
        "Values": [
            str(ano)
        ],
        "StorageDescriptor": {
            **storage_descriptor,
            "Location": (
                f"{SILVER_PATH}ano={ano}/"
            )
        }
    }

    try:

        glue.create_partition(
            DatabaseName=DATABASE,
            TableName=SILVER_TABLE,
            PartitionInput=partition_input
        )

        print(
            f"Partição {ano} criada."
        )

    except glue.exceptions.AlreadyExistsException:

        print(
            f"Partição {ano} já existe."
        )

{'ResponseMetadata': {'RequestId': '6f9e8b83-7cb7-45b1-958c-a29ac81fdf6a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 11 Aug 2026 01:33:13 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': '6f9e8b83-7cb7-45b1-958c-a29ac81fdf6a', 'cache-control': 'no-cache'}, 'RetryAttempts': 0}}
Partição 2023 criada.
{'ResponseMetadata': {'RequestId': 'a5f715ac-992f-4df5-babf-9f955780eab7', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 11 Aug 2026 01:33:13 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'a5f715ac-992f-4df5-babf-9f955780eab7', 'cache-control': 'no-cache'}, 'RetryAttempts': 0}}
Partição 2024 criada.
{'ResponseMetadata': {'RequestId': '2fddf469-8232-490b-89ef-0be4746149c8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 11 Aug 2026 01:33:13 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '2', 'conne

## 13. Validação final

In [ ]:
df_validacao = glueContext.create_data_frame_from_catalog(
    database=DATABASE,
    table_name=SILVER_TABLE
)

print("Registros lidos novamente da Silver:", df_validacao.count())
df_validacao.groupBy("ano").count().orderBy("ano").show()

Registros lidos novamente da Silver: 14002
+----+-----+
| ano|count|
+----+-----+
|2023| 5293|
|2024| 5215|
|2025| 3494|
+----+-----+
